# secp256k1_batch — GPU/CPU public-key generation

1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Upload this folder to `/content/sep3` (or clone the repo).
3. Run the cells below.

In [ ]:
# Confirm the GPU is visible.
!nvidia-smi

In [ ]:
# Install CuPy for the GPU backend (bundles the CUDA runtime; no nvcc needed).
!pip -q install cupy-cuda12x

In [ ]:
%cd /content/sep3

In [ ]:
import time
from secp256k1_batch import generate_public_keys, iter_keys
from secp256k1_batch import reference as R

# Smoke test: the first 3 public keys must match the trusted reference.
data = generate_public_keys(start=1, count=3, backend="auto")
assert list(iter_keys(data)) == R.public_keys_range(1, 3)
print("self-test OK:", [hex(p[0]) for p in iter_keys(data)][:1])

In [ ]:
# Benchmark: generate 1,000,000 keys on the GPU (uncompressed).
count = 1_000_000
t = time.perf_counter()
data = generate_public_keys(start=1, count=count, backend="gpu")
dt = time.perf_counter() - t
print(f"{count} keys in {dt:.3f}s = {count/dt/1e6:.2f} M keys/s")
print(f"output bytes: {len(data)}")

In [ ]:
# Compressed 33-byte SEC keys (parity prefix + x-coordinate).
data = generate_public_keys(start=1000, count=100_000, compressed=True, backend="gpu")
print(f"compressed bytes: {len(data)} (expected {100_000*33})")

## Tuning

* `keys_per_thread` (default 1024): larger amortizes the per-thread scalar
  multiplication; smaller increases parallelism. Try 512 / 2048 / 4096.
* `gpu_chunk` (default 4 Mi keys): bounds device memory; the range is
  processed in chunks.
* CPU-only: `backend="cpu", nthreads=<cores>`.

In [ ]:
from secp256k1_batch.engine import generate_public_keys as gen
for kpt in (512, 1024, 2048, 4096):
    t = time.perf_counter()
    gen(start=1, count=500_000, backend="gpu", keys_per_thread=kpt)
    dt = time.perf_counter() - t
    print(f"keys_per_thread={kpt:5d}: {500_000/dt/1e6:.2f} M keys/s")